# Testing Labeling process
using both global and rolling zscore

In [9]:
import os
import pandas as pd

In [11]:
daily = pd.read_parquet("../data/processed/chain/scam_daily_cleaned.parquet")

In [16]:
key_features = [
    'normal_sent_cnt', 'normal_recv_cnt', 'normal_total_cnt',
    'eth_sent_sum', 'eth_recv_sum', 'eth_net_flow',
    'uniq_peers_cnt', 'sessions_cnt', 'active_span_min', 'burst_max_tx_5m'
]

output_folder = "../data/per_address"
os.makedirs(output_folder, exist_ok=True)
# window_size = 30
# min_periods = 1
threshold = 2.0

# Exclude zero-activity days
zero_activity_mask = (daily[key_features].sum(axis=1) == 0)
excluded_days = zero_activity_mask.sum()

daily_filtered = daily.loc[~zero_activity_mask].copy()

print(f"Excluded {excluded_days} zero-activity days")
print(f"Remaining active days: {len(daily_filtered)}")

target_address = "0x3d80cd499dca1c6e41a1d20382ee6e1c70cea112"
addr_df = daily_filtered[daily_filtered["address"] == target_address].sort_values("day").copy()
addr_df["day"] = pd.to_datetime(addr_df["day"])

# Global Z-Scores
z_global = (addr_df[key_features] - addr_df[key_features].mean()) / (
    addr_df[key_features].std().replace(0, 1e-6)
)
addr_df["combined_z_score_global"] = z_global.abs().max(axis=1)

# # Rolling Z-Scores
# rolling_mean = addr_df[key_features].rolling(window=window_size, min_periods=min_periods).mean()
# rolling_std = addr_df[key_features].rolling(window=window_size, min_periods=min_periods).std().replace(0, 1e-6)
# z_rolling = (addr_df[key_features] - rolling_mean) / rolling_std
# addr_df["combined_z_score_rolling"] = z_rolling.abs().max(axis=1)

# Combined anomaly label
# Label as anomalous if either global OR rolling exceeds threshold
# addr_df["is_anomalous"] = (
#     (addr_df["combined_z_score_global"] > threshold) |
#     (addr_df["combined_z_score_rolling"] > threshold)
# ).astype(int)

addr_df["is_anomalous"] = (addr_df["combined_z_score_global"] > threshold).astype(int)

output_path = os.path.join(output_folder, f"{target_address}_z_score_combined_both.xlsx")
addr_df[["day"] + key_features + ["combined_z_score_global", "is_anomalous"]].to_excel(
    output_path, index=False
)


Excluded 2743336 zero-activity days
Remaining active days: 162699
